# ANEXO II — Códigos del Capítulo 4
## Interpretabilidad y análisis geoespacial

### Orden de ejecución
1. Ejecutar **PREPARACIÓN — Capítulo 4**.
2. Ejecutar A4.1, A4.2, …, A4.25 **en orden y sin saltarse celdas**.
3. A4.16 y A4.25 son las versiones definitivas corregidas.

> **Nota:** el notebook necesita que `dataset_modelo_hortaleza.csv` esté en la misma carpeta que el `.ipynb`. La celda de preparación reconstruye las variables del capítulo 3 que utiliza el capítulo 4, incluido `modelo_final`.

In [ ]:
# PREPARACIÓN — Capítulo 4
# Ejecutar esta celda antes de A4.1.
#
# Esta celda reproduce las variables necesarias del Modelo B del capítulo 3.
# El Modelo B utiliza exactamente las 20 variables definidas en A3.28:
# 10 variables estructurales + 2 geográficas + 8 variables de distancia.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import RepeatedKFold, GroupKFold, cross_val_score

from xgboost import XGBRegressor

# Dataset final utilizado para la modelización en el capítulo 3.
RUTA_DATASET = "dataset_modelo_hortaleza.csv"
df = pd.read_csv(RUTA_DATASET)

# Variable objetivo
y = df["price"]

# Variables del Modelo B (A3.28 del Anexo II)
variables_modelo_a = [
    "size", "rooms", "bathrooms", "floor", "hasLift",
    "status", "propertyType", "typology", "exterior",
    "newDevelopment"
]

variables_modelo_b = variables_modelo_a + [
    "latitude", "longitude",
    "dist_metro", "dist_bus", "dist_school", "dist_hospital",
    "dist_health_center", "dist_supermarket", "dist_park",
    "dist_sports"
]

# Comprobación del dataset.
faltantes = [v for v in variables_modelo_b + ["price"] if v not in df.columns]
if faltantes:
    raise ValueError(
        "El dataset no contiene estas variables necesarias: "
        + ", ".join(faltantes)
    )

X_a = df[variables_modelo_a].copy()
X_b = df[variables_modelo_b].copy()

# Variables categóricas del Modelo B (A3.30).
variables_categoricas_a = [
    "floor", "hasLift", "status", "propertyType",
    "typology", "exterior"
]
variables_categoricas_b = variables_categoricas_a.copy()

variables_numericas_b = [
    v for v in variables_modelo_b
    if v not in variables_categoricas_b
]

# Preprocesamiento equivalente al utilizado en A3.30.
preprocesador_b = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            variables_categoricas_b
        )
    ],
    remainder="passthrough"
)

# XGBoost B (A3.38).
modelo_xgb_b = Pipeline([
    (
        "preprocesamiento",
        preprocesador_b
    ),
    (
        "modelo",
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    )
])

# Entrenamiento final con las 561 viviendas disponibles (A3.62).
modelo_xgb_b.fit(X_b, y)

# Variable utilizada por todos los códigos A4.
modelo_final = modelo_xgb_b

print("Entorno del capítulo 4 preparado correctamente.")
print("Dimensiones del dataset:", df.shape)
print("Modelo final: XGBoost B")
print("Número de viviendas utilizadas:", len(X_b))
print("Número de variables predictoras:", X_b.shape[1])


In [ ]:
# A4.1 — Importancia de variables transformadas

preprocesador_final = modelo_final.named_steps["preprocesamiento"]

modelo_xgb_final = modelo_final.named_steps["modelo"]

nombres_variables = preprocesador_final.get_feature_names_out()

importancias = modelo_xgb_final.feature_importances_

df_importancias_transformadas = pd.DataFrame({ "Variable":
nombres_variables, "Importancia": importancias

}).sort_values("Importancia", ascending=False).reset_index(drop=True)

print("Dimensiones del dataset:", df.shape)

print("Modelo final: XGBoost B")

print("Número de viviendas utilizadas:", len(X_b))

print("Número de variables transformadas:", len(nombres_variables))

print("Número de importancias:", len(importancias))

display(df_importancias_transformadas.head(20))


In [ ]:
# A4.2 — Agregación por variable original

def obtener_variable_original(nombre):

    for variable in variables_categoricas_b:

        prefijo = f"categoricas__{variable}_"

        if nombre.startswith(prefijo):

            return variable

    if nombre.startswith("remainder__"):

        return nombre.replace("remainder__", "")

    return nombre



df_importancias_transformadas["Variable_original"] = ( df_importancias_transformadas["Variable"].apply(obtener_variable_original)

)

df_importancias = ( df_importancias_transformadas.groupby("Variable_original", as_index=False)["Importancia"]


.sum().sort_values("Importancia", ascending=False).reset_index(drop=True)

)

display(df_importancias)


In [ ]:
# A4.3 — Gráfico de importancia

df_grafico_importancias = df_importancias.sort_values("Importancia", ascending=True)

plt.figure(figsize=(10, 8))

plt.barh(df_grafico_importancias["Variable_original"], df_grafico_importancias["Importancia"])

plt.xlabel("Importancia de la variable")

plt.ylabel("Variable")

plt.title("Importancia de las variables en el modelo XGBoost B")

plt.tight_layout()

plt.show()


In [ ]:
# A4.4 — Porcentaje y porcentaje acumulado

df_importancias_final = df_importancias.copy()

df_importancias_final["Importancia (%)"] = df_importancias_final["Importancia"] * 100

df_importancias_final["Importancia acumulada (%)"] = ( df_importancias_final["Importancia (%)"].cumsum()

)

df_importancias_final["Importancia (%)"] = df_importancias_final["Importancia (%)"].round(2)

df_importancias_final["Importancia acumulada (%)"] = ( df_importancias_final["Importancia acumulada (%)"].round(2)

)

display(df_importancias_final)


In [ ]:
# A4.5 — Importancia conjunta por grupos

variables_geograficas = ["latitude", "longitude"]

variables_entorno = [ "dist_metro", "dist_bus", "dist_school", "dist_hospital", "dist_health_center", "dist_supermarket", "dist_park", "dist_sports"

]

variables_estructurales = [ v for v in variables_modelo_b

    if v not in variables_geograficas + variables_entorno

]

importancia_estructural = df_importancias_final.loc[ df_importancias_final["Variable_original"].isin(variables_estructurales), "Importancia"

].sum()

importancia_geografica = df_importancias_final.loc[ df_importancias_final["Variable_original"].isin(variables_geograficas), "Importancia"

].sum()

importancia_entorno = df_importancias_final.loc[ df_importancias_final["Variable_original"].isin(variables_entorno), "Importancia"

].sum()

df_importancia_grupos = pd.DataFrame({ "Grupo": ["Variables estructurales", "Variables geográficas", "Variables de entorno"], "Importancia":
[importancia_estructural, importancia_geografica, importancia_entorno]

})

df_importancia_grupos["Importancia (%)"] = ( df_importancia_grupos["Importancia"] * 100

).round(2)

display(df_importancia_grupos)


In [ ]:
# A4.6 — Gráfico de importancia por grupos

plt.figure(figsize=(9, 6))

plt.bar(df_importancia_grupos["Grupo"], df_importancia_grupos["Importancia (%)"])

plt.ylabel("Importancia (%)")

plt.xlabel("Grupo de variables")

plt.title("Importancia conjunta de los grupos de variables en el modelo XGBoost B")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()


In [ ]:
# A4.7 — Preparación de SHAP

import shap

X_b_transformado = modelo_final.named_steps["preprocesamiento"].transform(X_b)

nombres_shap = modelo_final.named_steps["preprocesamiento"].get_feature_names_out()

print("Datos preparados para SHAP.")

print("Número de observaciones:", X_b_transformado.shape[0])

print("Número de variables transformadas:", X_b_transformado.shape[1])

print(nombres_shap[:20])


In [ ]:
# A4.8 — Cálculo de valores SHAP

modelo_xgb_shap = modelo_final.named_steps["modelo"]

explainer = shap.TreeExplainer(modelo_xgb_shap)

shap_values = explainer.shap_values(X_b_transformado)

print("Valores SHAP calculados correctamente.")

print("Dimensiones de los valores SHAP:", shap_values.shape)

print("Dimensiones de los datos transformados:", X_b_transformado.shape)

print("Número de variables:", len(nombres_shap))

print("Coinciden las dimensiones:", shap_values.shape == X_b_transformado.shape)


In [ ]:
# A4.9 — Importancia SHAP por variable original

df_shap = pd.DataFrame(shap_values, columns=nombres_shap)

df_shap_importancia = pd.DataFrame({ "Variable": nombres_shap, "Importancia_SHAP":
np.abs(shap_values).mean(axis=0)

})

df_shap_importancia["Variable_original"] = ( df_shap_importancia["Variable"].apply(obtener_variable_original)

)

df_shap_importancia_original = ( df_shap_importancia.groupby("Variable_original", as_index=False)["Importancia_SHAP"]


.sum().sort_values("Importancia_SHAP", ascending=False).reset_index(drop=True)

)

display(df_shap_importancia_original)


In [ ]:
# A4.10 — Gráfico de importancia SHAP

df_shap_grafico = df_shap_importancia_original.sort_values("Importancia_SHAP", ascending=True)

plt.figure(figsize=(10, 8))

plt.barh(df_shap_grafico["Variable_original"], df_shap_grafico["Importancia_SHAP"])

plt.xlabel("Importancia SHAP media absoluta (€)")

plt.ylabel("Variable")

plt.title("Importancia global de las variables según SHAP")

plt.tight_layout()

plt.show()


In [53]:
# A4.11 — SHAP agrupado en 20 variables, conservando signo

df_shap_transformado = pd.DataFrame(shap_values, columns=nombres_shap)

df_shap_original = pd.DataFrame(index=df_shap_transformado.index)



for variable in variables_categoricas_b:

    columnas_variable = [ c for c in nombres_shap

        if c.startswith(f"categoricas__{variable}_")

    ]

    df_shap_original[variable] = df_shap_transformado[columnas_variable].sum(axis=1)



variables_numericas_b = [ "size", "rooms", "bathrooms", "newDevelopment", "latitude", "longitude", "dist_metro", "dist_bus", "dist_school", "dist_hospital", "dist_health_center", "dist_supermarket", "dist_park", "dist_sports"

]

for variable in variables_numericas_b:

    df_shap_original[variable] = df_shap_transformado[f"remainder__{variable}"]

df_shap_original = df_shap_original[variables_modelo_b]

print(df_shap_original.shape)

display(df_shap_original.head())


(561, 20)


,size,rooms,bathrooms,floor,hasLift,status,propertyType,typology,exterior,newDevelopment,latitude,longitude,dist_metro,dist_bus,dist_school,dist_hospital,dist_health_center,dist_supermarket,dist_park,dist_sports
0,-176008.906250,-13682.324219,56136.832031,-13152.576172,147.079346,6321.061035,25245.207031,-145.018982,-14878.518555,-672.211975,-44166.957031,6225.270508,20376.982422,21522.468750,10034.285156,2590.626465,6820.188965,-12938.286133,-987.537231,-8707.677734
1,-130363.906250,-7108.528320,-39847.101562,-7307.273926,2124.778320,-41115.207031,-4210.772461,-393.156555,-1552.415771,-646.134277,148773.046875,35011.402344,14577.227539,-27555.718750,18680.669922,-1126.443359,14352.723633,-2997.780762,8372.134766,-2809.498535
2,126752.234375,-23090.460938,310260.187500,10024.162109,2439.410889,10133.215820,19650.001953,1134.085205,2380.023438,-618.700012,40042.207031,-100471.953125,26486.681641,-182609.640625,-8479.802734,-137720.078125,-3048.206543,-4917.897949,-10736.297852,-18985.121094
3,919185.437500,14102.612305,531958.250000,12028.168945,1500.135254,32721.986328,21109.335938,2033.018555,2972.531738,-730.531982,78537.382812,60425.316406,3696.277344,144996.765625,27130.060547,62983.421875,10965.531250,-31574.023438,8720.448242,17710.617188
4,-221149.953125,-21204.072266,127573.476562,-520.230774,1160.619385,5503.640137,3329.772217,825.185913,26879.728516,-1385.734497,110250.984375,-1387.732910,-25423.500000,2042.485229,-21077.695312,10557.439453,-25930.220703,-18094.724609,16438.951172,-24077.681641


In [54]:
# A4.12 — Dirección media de las contribuciones SHAP

df_direccion_shap = pd.DataFrame({ "Variable":
df_shap_original.columns, "SHAP medio (€)":
df_shap_original.mean().values, "SHAP mediano (€)":
df_shap_original.median().values, "SHAP medio absoluto (€)":
df_shap_original.abs().mean().values

}).sort_values("SHAP medio absoluto (€)", ascending=False).reset_index(drop=True)

for c in ["SHAP medio (€)", "SHAP mediano (€)", "SHAP medio absoluto (€)"]:

    df_direccion_shap[c] = df_direccion_shap[c].round(2)

display(df_direccion_shap)


,Variable,SHAP medio (€),SHAP mediano (€),SHAP medio absoluto (€)
0,size,-5388.419922,-495625.187500,759372.625000
1,bathrooms,18026.000000,-74075.656250,178181.640625
2,latitude,-11649.080078,-26253.679688,59161.980469
3,dist_bus,-16825.839844,-29762.449219,50520.539062
4,dist_metro,15111.099609,22192.800781,45596.699219
5,longitude,3923.979980,5384.250000,27969.439453
6,dist_hospital,4001.879883,5762.740234,26076.689453
7,rooms,-3408.010010,-11117.780273,25123.070312
8,status,80.769997,8345.160156,23126.359375
9,dist_supermarket,-10876.570312,-9318.330078,19932.689453


In [ ]:
# A4.13 — Relación entre variables espaciales y SHAP

variables_espaciales_relevantes = ["latitude", "dist_bus", "dist_metro", "longitude"]

for variable in variables_espaciales_relevantes:

    plt.figure(figsize=(9, 6))

    plt.scatter(X_b[variable], df_shap_original[variable], alpha=0.6, s=25)

    plt.axhline(0, linewidth=1)

    plt.xlabel(variable)

    plt.ylabel("Contribución SHAP (€)")

    plt.title(f"Relación entre {variable} y su contribución SHAP")

    plt.tight_layout()

    plt.show()


In [ ]:
# A4.14 — SHAP Summary Plot

variables_numericas_ordenadas = ( df_shap_importancia_original[ df_shap_importancia_original["Variable_original"].isin(variables_numericas_b)


].sort_values("Importancia_SHAP", ascending=False)["Variable_original"].tolist()

)

variables_numericas_top = variables_numericas_ordenadas[:10]

shap_numericas = df_shap_original[variables_numericas_top].values

X_numericas = X_b[variables_numericas_top].copy()

plt.figure(figsize=(10, 8))

shap.summary_plot( shap_numericas, X_numericas, feature_names=variables_numericas_top, show=False

)

plt.title("SHAP Summary Plot de las principales variables numéricas")

plt.tight_layout()

plt.show()


In [ ]:
# A4.15 — Comparación SHAP entre valores bajos y altos

variables_espaciales_analisis = ["latitude", "dist_bus", "dist_metro", "longitude"]

resultados_espaciales_shap = []

for variable in variables_espaciales_analisis:

    valores = X_b[variable]

    contribuciones = df_shap_original[variable]

    q25, q75 = valores.quantile(0.25), valores.quantile(0.75)

    grupo_bajo, grupo_alto = valores <= q25, valores >= q75

    shap_bajo, shap_alto = contribuciones[grupo_bajo].mean(), contribuciones[grupo_alto].mean()

    resultados_espaciales_shap.append({ "Variable": variable, "Percentil 25": q25, "SHAP medio valores bajos (€)": shap_bajo, "Percentil 75": q75, "SHAP medio valores altos (€)": shap_alto, "Diferencia SHAP (€)":
shap_alto - shap_bajo

    })

df_resultados_espaciales_shap = pd.DataFrame(resultados_espaciales_shap)

display(df_resultados_espaciales_shap.round(2))


In [ ]:
# A4.16 — Importancia SHAP conjunta por grupos (versión definitiva)

df_shap_importancia_original_def = pd.DataFrame({ "Variable":
df_shap_original.columns, "Importancia_SHAP":
df_shap_original.abs().mean(axis=0).values

})

variables_geograficas = ["latitude", "longitude"]

variables_entorno = [ "dist_metro", "dist_bus", "dist_school", "dist_hospital", "dist_health_center", "dist_supermarket", "dist_park", "dist_sports"

]

variables_estructurales = [ v for v in variables_modelo_b

    if v not in variables_geograficas + variables_entorno

]

shap_estructural = df_shap_importancia_original_def.loc[ df_shap_importancia_original_def["Variable"].isin(variables_estructurales), "Importancia_SHAP"

].sum()

shap_geografica = df_shap_importancia_original_def.loc[ df_shap_importancia_original_def["Variable"].isin(variables_geograficas), "Importancia_SHAP"

].sum()

shap_entorno = df_shap_importancia_original_def.loc[ df_shap_importancia_original_def["Variable"].isin(variables_entorno), "Importancia_SHAP"

].sum()

df_shap_grupos = pd.DataFrame({ "Grupo": ["Variables estructurales", "Variables geográficas", "Variables de entorno"], "Importancia SHAP (€)":
[shap_estructural, shap_geografica, shap_entorno]

})

df_shap_grupos["Importancia SHAP (%)"] = ( df_shap_grupos["Importancia SHAP (€)"] / df_shap_grupos["Importancia SHAP (€)"].sum() * 100

)

display(df_shap_grupos.round(2))


In [ ]:
# A4.17 — Predicciones y errores espaciales

predicciones_finales = modelo_final.predict(X_b)

df_espacial = df.copy()

df_espacial["price_predicho"] = predicciones_finales

df_espacial["error"] = df_espacial["price"] - df_espacial["price_predicho"]

df_espacial["error_absoluto"] = df_espacial["error"].abs()

print("Número de viviendas:", len(df_espacial))

print("Precio observado medio:", round(df_espacial["price"].mean(), 2), "€")

print("Precio predicho medio:", round(df_espacial["price_predicho"].mean(), 2), "€")

print("Error medio:", round(df_espacial["error"].mean(), 2), "€")

print("MAE sobre las 561 viviendas:", round(df_espacial["error_absoluto"].mean(), 2), "€")

print("Error mínimo:", round(df_espacial["error"].min(), 2), "€")

print("Error máximo:", round(df_espacial["error"].max(), 2), "€")


In [ ]:
# A4.18 — Mapa exploratorio de errores

import geopandas as gpd

limite_hortaleza_gdf = gpd.read_file("Dist_Hortaleza.shp")

limite_error = np.max(np.abs(df_espacial["error"]))

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter( df_espacial["longitude"], df_espacial["latitude"], c=df_espacial["error"], cmap="coolwarm", vmin=-limite_error, vmax=limite_error, s=25, alpha=0.8

)

limite_hortaleza_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=1.5)

cbar = plt.colorbar(scatter, ax=ax)

cbar.set_label("Error de predicción (€)")

ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")

ax.set_title("Distribución espacial de los errores del modelo XGBoost B")

plt.tight_layout(); plt.show()


In [ ]:
# A4.19 — Mapa de predicciones

precio_min = df_espacial["price_predicho"].min()

precio_max = df_espacial["price_predicho"].max()

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter( df_espacial["longitude"], df_espacial["latitude"], c=df_espacial["price_predicho"], cmap="viridis", vmin=precio_min, vmax=precio_max, s=25, alpha=0.8

)

limite_hortaleza_gdf.to_crs("EPSG:4326").boundary.plot(ax=ax, linewidth=1.5)

cbar = plt.colorbar(scatter, ax=ax)

cbar.set_label("Precio predicho (€)")

ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")

ax.set_title("Distribución espacial de las predicciones del modelo XGBoost B")

plt.tight_layout(); plt.show()


In [ ]:
# A4.20 — I de Moran exploratorio

from libpysal.weights import KNN

from esda.moran import Moran

gdf_errores = gpd.GeoDataFrame( df_espacial.copy(), geometry=gpd.points_from_xy( df_espacial["longitude"], df_espacial["latitude"]

    ), crs="EPSG:4326"

).to_crs("EPSG:25830")

coordenadas_25830 = np.column_stack([ gdf_errores.geometry.x, gdf_errores.geometry.y

])

pesos_knn = KNN.from_array(coordenadas_25830, k=8)

pesos_knn.transform = "R"

moran_errores = Moran(gdf_errores["error"].values, pesos_knn, permutations=999)

print("I de Moran:", moran_errores.I)

print("Esperanza bajo H0:", moran_errores.EI)

print("Z-score:", moran_errores.z_sim)

print("P-valor:", moran_errores.p_sim)


In [ ]:
# A4.21 — Sensibilidad de Moran

resultados_moran = []

for k in [8, 12, 15]:

    pesos = KNN.from_array(coordenadas_25830, k=k)

    pesos.transform = "R"

    moran = Moran(gdf_errores["error"].values, pesos, permutations=999)

    resultados_moran.append({ "K vecinos": k, "Componentes desconectados": pesos.n_components, "I de Moran": moran.I, "Z-score": moran.z_sim, "P-valor": moran.p_sim

    })

df_sensibilidad_moran = pd.DataFrame(resultados_moran)

display(df_sensibilidad_moran.round(6))


In [ ]:
# A4.22 — Moran definitivo con K=12

pesos_moran_final = KNN.from_array(coordenadas_25830, k=12)

pesos_moran_final.transform = "R"

moran_final = Moran( gdf_errores["error"].values, pesos_moran_final, permutations=999

)

df_moran_final = pd.DataFrame({ "Indicador": [ "Número de vecinos (K)", "Componentes conectados", "I de Moran", "Esperanza bajo H0", "Z-score", "P-valor por permutaciones"

    ], "Resultado": [ 12, pesos_moran_final.n_components, moran_final.I, moran_final.EI, moran_final.z_sim, moran_final.p_sim

    ]

})

display(df_moran_final)


In [ ]:
# A4.23 — Errores fuera de muestra y Moran

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



X_train, X_test, y_train, y_test = train_test_split( X_b, y, test_size=0.20, random_state=42

)



modelo_xgb_test = XGBRegressor( n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1

)

modelo_test = Pipeline(steps=[ ("preprocesamiento", ColumnTransformer( transformers=[("categoricas", OneHotEncoder( handle_unknown="ignore", sparse_output=False

        ), variables_categoricas_b)], remainder="passthrough"

    )), ("modelo", modelo_xgb_test)

])

modelo_test.fit(X_train, y_train)

predicciones_test = modelo_test.predict(X_test)

errores_test = y_test.values - predicciones_test



print("Viviendas de entrenamiento:", len(X_train))

print("Viviendas de prueba:", len(X_test))

print("MAE:", mean_absolute_error(y_test, predicciones_test))

print("RMSE:", np.sqrt(mean_squared_error(y_test, predicciones_test)))

print("R²:", r2_score(y_test, predicciones_test))



gdf_test = gpd.GeoDataFrame( X_test.copy(), geometry=gpd.points_from_xy(X_test["longitude"], X_test["latitude"]), crs="EPSG:4326"

).to_crs("EPSG:25830")

gdf_test["price_observado"] = y_test.values

gdf_test["price_predicho"] = predicciones_test

gdf_test["error"] = errores_test

gdf_test["error_absoluto"] = np.abs(errores_test)



coordenadas_test = np.column_stack([ gdf_test.geometry.x, gdf_test.geometry.y

])

pesos_test = KNN.from_array(coordenadas_test, k=12)

pesos_test.transform = "R"

moran_test = Moran(gdf_test["error"].values, pesos_test, permutations=999)



print("I de Moran:", moran_test.I)

print("Esperanza bajo H0:", moran_test.EI)

print("Z-score:", moran_test.z_sim)

print("P-valor:", moran_test.p_sim)

print("Componentes conectados:", pesos_test.n_components)


In [ ]:
# A4.24 — Sensibilidad de Moran en el conjunto de prueba

resultados_moran_test = []

for k in [8, 12, 15]:

    pesos = KNN.from_array(coordenadas_test, k=k)

    pesos.transform = "R"

    moran = Moran(gdf_test["error"].values, pesos, permutations=999)

    resultados_moran_test.append({ "K vecinos": k, "Componentes conectados": pesos.n_components, "I de Moran": moran.I, "Z-score": moran.z_sim, "P-valor": moran.p_sim

    })

df_sensibilidad_moran_test = pd.DataFrame(resultados_moran_test)

display(df_sensibilidad_moran_test.round(6))


In [ ]:
# A4.25 — Mapa definitivo de errores del conjunto de prueba

gdf_test_mapa = gdf_test.to_crs("EPSG:4326")

limite_hortaleza_test = limite_hortaleza_gdf.to_crs("EPSG:4326")

limite_error_test = np.max(np.abs(gdf_test_mapa["error"]))



fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter( gdf_test_mapa.geometry.x, gdf_test_mapa.geometry.y, c=gdf_test_mapa["error"], cmap="coolwarm", vmin=-limite_error_test, vmax=limite_error_test, s=35, alpha=0.85

)

limite_hortaleza_test.boundary.plot(ax=ax, linewidth=1.5)

cbar = plt.colorbar(scatter, ax=ax)

cbar.set_label("Error de predicción (€)")

ax.set_xlabel("Longitud")

ax.set_ylabel("Latitud")

ax.set_title( "Errores de predicción del modelo XGBoost B "

    "en el conjunto de prueba"

)

plt.tight_layout()

plt.show()
